## 01. Create Discovery Golden Dataset

### Why are we implementing this?

We need a separate golden dataset to evaluate whether SEVA_AI can identify relevant government schemes from a citizen's real-world problem.

Unlike our existing FAQ dataset, citizen queries must not contain scheme names.

Each query will have manually verified relevance judgments for multiple schemes.

We will use this dataset to evaluate:
- Scheme discovery accuracy
- Recall@K and Precision@K
- MRR and MAP
- nDCG with graded relevance

We will initially target 150 citizen queries.

In [1]:
import json
from pathlib import Path

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

DISCOVERY_DATASET_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "golden_dataset"
    / "discovery"
)

DISCOVERY_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_golden_dataset.jsonl"
)

SCHEMA_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_schema.json"
)

schema = {
    "query_id": "Unique query identifier",
    "citizen_query": "Citizen's actual problem without scheme names",
    "category": "Government scheme category",
    "answerable": "Boolean",
    "relevant_scheme_ids": [
        "Verified relevant scheme IDs"
    ],
    "relevance_judgments": [
        {
            "scheme_id": "Government scheme ID",
            "relevance_grade": 2,
            "rationale": "Why this scheme is relevant",
            "source_url": "Official verification URL"
        }
    ]
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

if not DATASET_PATH.exists():
    DATASET_PATH.touch()

print("Discovery benchmark initialized!")
print("Dataset:", DATASET_PATH)
print("Schema:", SCHEMA_PATH)
print("Target queries: 150")

Discovery benchmark initialized!
Dataset: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_golden_dataset.jsonl
Schema: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_schema.json
Target queries: 150


## 02. Analyze Scheme Categories

### Why are we implementing this?

We need to understand the distribution of government schemes before creating our 150 citizen queries.

We will inspect the existing scheme dataset to identify:
- Available scheme categories
- Number of schemes per category
- Available metadata fields
- Sample scheme records

This analysis will help us create a balanced discovery benchmark across different government domains.

In [3]:
import json
from collections import Counter
from pathlib import Path

# SCHEME_PATH = (
#     PROJECT_ROOT
#     / "notebooks"
#     / "golden_dataset"
#     / "selected_schemes.json"
# )

SCHEME_PATH ="G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\data\\analysis\\all_schemes.json"

with open(SCHEME_PATH, "r", encoding="utf-8") as f:
    scheme_data = json.load(f)

if isinstance(scheme_data, list):
    schemes = scheme_data
elif isinstance(scheme_data, dict):
    schemes = scheme_data.get("schemes", [])
else:
    raise ValueError("Unexpected scheme dataset structure")

print("Total schemes:", len(schemes))

if not schemes:
    raise ValueError("No scheme records found.")

print("\nAvailable fields:")
print(list(schemes[0].keys()))

category_counts = Counter()

for scheme in schemes:
    categories = scheme.get("categories", [])

    if isinstance(categories, str):
        categories = [categories]

    category_counts.update(
        category for category in categories if category
    )

print("\nCategory distribution:")

for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nSample scheme:")
print(json.dumps(schemes[0], indent=2, ensure_ascii=False))

Total schemes: 4428

Available fields:
['scheme_id', 'internal_id', 'slug', 'name', 'short_title', 'level', 'ministry', 'nodal_department', 'department', 'states', 'categories', 'tags', 'target_beneficiaries', 'status', 'open_date', 'close_date', 'brief_description', 'details', 'benefits', 'exclusions', 'eligibility', 'application_process', 'documents_required', 'faqs', 'apply_url', 'apply_links', 'official_links', 'myscheme_url', 'source', 'fetched_on']

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Sample scheme:
{
  "scheme_id": "LBezF5gB5uF80_MGGY81",
  "internal_id": "628727

## Full Scheme Dataset Validation

Validate the complete government scheme dataset before building the citizen-discovery benchmark. This checks data integrity, category coverage, scheme status, and geographic distribution.

In [7]:
import json
from pathlib import Path
from collections import Counter

SCHEME_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\data\analysis\all_schemes.json")

with open(SCHEME_PATH, "r", encoding="utf-8") as file:
    schemes = json.load(file)

print("Total schemes:", len(schemes))
print("Unique scheme IDs:", len({s.get("scheme_id") for s in schemes}))
print("Categories:", len({
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
}))

category_counts = Counter(
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
)

status_counts = Counter(
    scheme.get("status", "unknown")
    for scheme in schemes
)

level_counts = Counter(
    scheme.get("level", "unknown")
    for scheme in schemes
)

state_counts = Counter(
    state
    for scheme in schemes
    for state in scheme.get("states", [])
)

print("\nCategory distribution:")
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nStatus distribution:", status_counts)
print("\nLevel distribution:", level_counts)
print("\nTop 15 states:", state_counts.most_common(15))

Total schemes: 4428
Unique scheme IDs: 4428
Categories: 15

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Status distribution: Counter({'active': 4384, 'closed': 44})

Level distribution: Counter({'State': 3735, 'Central': 693})

Top 15 states: [('All', 673), ('Gujarat', 594), ('Uttarakhand', 416), ('Madhya Pradesh', 263), ('Goa', 247), ('Puducherry', 245), ('Haryana', 224), ('Tamil Nadu', 212), ('Rajasthan', 147), ('West Bengal', 105), ('Chhattisgarh', 103), ('Bihar', 102), ('Assam', 86), ('Jharkhand', 86), ('Meghalaya', 85)]


## Geographic Coverage Analysis

Analyze scheme availability by state and scheme level. This helps us design citizen queries that test both Central and State government scheme discovery, including location-specific eligibility.

In [8]:
from collections import Counter

central_schemes = [
    s for s in schemes if s.get("level") == "Central"
]

state_schemes = [
    s for s in schemes if s.get("level") == "State"
]

state_distribution = Counter(
    state
    for scheme in state_schemes
    for state in scheme.get("states", [])
    if state != "All"
)

print("Central schemes:", len(central_schemes))
print("State schemes:", len(state_schemes))

print("\nState-wise scheme distribution:")
for state, count in state_distribution.most_common():
    print(f"{state}: {count}")

print("\nCentral scheme state values:")
print(Counter(
    state
    for scheme in central_schemes
    for state in scheme.get("states", [])
))

Central schemes: 693
State schemes: 3735

State-wise scheme distribution:
Gujarat: 592
Uttarakhand: 415
Madhya Pradesh: 261
Goa: 246
Puducherry: 245
Haryana: 224
Tamil Nadu: 206
Rajasthan: 145
West Bengal: 104
Chhattisgarh: 101
Bihar: 101
Jharkhand: 85
Meghalaya: 79
Kerala: 78
Assam: 77
Maharashtra: 77
Odisha: 77
Himachal Pradesh: 73
Dadra & Nagar Haveli and Daman & Diu: 59
Karnataka: 54
Delhi: 49
Andhra Pradesh: 44
Punjab: 41
Uttar Pradesh: 41
Jammu and Kashmir: 39
Tripura: 36
Arunachal Pradesh: 35
Manipur: 24
Sikkim: 21
Mizoram: 21
Telangana: 21
Chandigarh: 19
Andaman and Nicobar Islands: 17
Nagaland: 16
Lakshadweep: 9
Ladakh: 3

Central scheme state values:
Counter({'All': 673, 'Assam': 9, 'Tripura': 7, 'Arunachal Pradesh': 7, 'Manipur': 7, 'Mizoram': 7, 'Tamil Nadu': 6, 'Sikkim': 6, 'Meghalaya': 6, 'Kerala': 6, 'Nagaland': 6, 'Karnataka': 4, 'Jammu and Kashmir': 3, 'Gujarat': 2, 'Rajasthan': 2, 'Madhya Pradesh': 2, 'Chhattisgarh': 2, 'Odisha': 2, 'Maharashtra': 1, 'West Bengal': 1,

## National Geographic Coverage Inventory

Prepare a complete state and union territory coverage inventory from the 4,428-scheme dataset. This will help distribute the citizen-discovery benchmark across India without prioritizing one state.

In [9]:
from collections import Counter

state_scheme_counts = Counter(
    state
    for scheme in schemes
    if scheme.get("level") == "State"
    for state in scheme.get("states", [])
    if state != "All"
)

print("Total states and UTs:", len(state_scheme_counts))
print("\nNational geographic coverage:\n")

for index, (state, count) in enumerate(
    sorted(state_scheme_counts.items()), start=1
):
    print(f"{index:02d}. {state}: {count} schemes")

Total states and UTs: 36

National geographic coverage:

01. Andaman and Nicobar Islands: 17 schemes
02. Andhra Pradesh: 44 schemes
03. Arunachal Pradesh: 35 schemes
04. Assam: 77 schemes
05. Bihar: 101 schemes
06. Chandigarh: 19 schemes
07. Chhattisgarh: 101 schemes
08. Dadra & Nagar Haveli and Daman & Diu: 59 schemes
09. Delhi: 49 schemes
10. Goa: 246 schemes
11. Gujarat: 592 schemes
12. Haryana: 224 schemes
13. Himachal Pradesh: 73 schemes
14. Jammu and Kashmir: 39 schemes
15. Jharkhand: 85 schemes
16. Karnataka: 54 schemes
17. Kerala: 78 schemes
18. Ladakh: 3 schemes
19. Lakshadweep: 9 schemes
20. Madhya Pradesh: 261 schemes
21. Maharashtra: 77 schemes
22. Manipur: 24 schemes
23. Meghalaya: 79 schemes
24. Mizoram: 21 schemes
25. Nagaland: 16 schemes
26. Odisha: 77 schemes
27. Puducherry: 245 schemes
28. Punjab: 41 schemes
29. Rajasthan: 145 schemes
30. Sikkim: 21 schemes
31. Tamil Nadu: 206 schemes
32. Telangana: 21 schemes
33. Tripura: 36 schemes
34. Uttar Pradesh: 41 schemes
35. 

## National Discovery Benchmark Allocation

Create a balanced allocation of 150 citizen queries across all 15 scheme categories and 36 Indian States and Union Territories. Include both Central and State scheme scenarios while ensuring every state and UT is represented.

One important distinction: 10% is our sampling target, not necessarily one unique question per scheme. We should generate questions from scheme information, then verify that each question has the correct relevant scheme IDs. Some citizen questions may legitimately match multiple schemes.

In [10]:
import math
import pandas as pd

state_targets = pd.DataFrame([
    {
        "state": state,
        "scheme_count": count,
        "target_questions": math.ceil(count * 0.10)
    }
    for state, count in state_scheme_counts.items()
])

central_count = len(central_schemes)
central_target = math.ceil(central_count * 0.10)

print(state_targets.to_string(index=False))
print("\nState questions:", state_targets["target_questions"].sum())
print("Central questions:", central_target)
print(
    "Total target questions:",
    state_targets["target_questions"].sum() + central_target
)

                               state  scheme_count  target_questions
                               Delhi            49                 5
                              Sikkim            21                 3
                             Haryana           224                23
                           Karnataka            54                 6
                             Mizoram            21                 3
                    Himachal Pradesh            73                 8
                             Gujarat           592                60
                               Assam            77                 8
                           Jharkhand            85                 9
                            Nagaland            16                 2
                             Manipur            24                 3
                           Rajasthan           145                15
                         West Bengal           104                11
                         Maharasht

## Discovery Benchmark Scheme Sampling

Select approximately 10% of State schemes for each State/UT and 10% of Central schemes. Each selected scheme becomes a source for generating one natural-language citizen query. Preserve the original scheme information and geographic scope for relevance judgments.

In [10]:
import json
import math
import random
from pathlib import Path

random.seed(42)

discovery_dir = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)
discovery_dir.mkdir(parents=True, exist_ok=True)

generation_tasks = []

# Sample State schemes separately for each State/UT
for state, count in state_scheme_counts.items():
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Sample Central schemes
central_target = math.ceil(len(central_schemes) * 0.10)
selected_central = random.sample(
    central_schemes, central_target
)

for scheme in selected_central:
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

output_path = discovery_dir / "discovery_generation_tasks.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    for task in generation_tasks:
        file.write(
            json.dumps(task, ensure_ascii=False) + "\n"
        )

print("Generation tasks:", len(generation_tasks))
print("State tasks:", sum(t["scope"] == "State" for t in generation_tasks))
print("Central tasks:", sum(t["scope"] == "Central" for t in generation_tasks))
print("Saved:", output_path)

Generation tasks: 462
State tasks: 392
Central tasks: 70
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_generation_tasks.jsonl


In [11]:
import math
import random

random.seed(42)

generation_tasks = []

# State scheme sampling
for state in state_scheme_counts:
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Central scheme sampling
central_target = math.ceil(len(central_schemes) * 0.10)

for scheme in random.sample(central_schemes, central_target):
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

print("Total generation tasks:", len(generation_tasks))

Total generation tasks: 462


In [16]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

# google_api = os.getenv("GOOGLE_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")

if groq_api is None:
    raise ValueError("GROQ_API_KEY is not set in the environment variables")

llm = ChatGroq(
    api_key=groq_api,
    model="openai/gpt-oss-120b"
    # temperature=0.7,
    # max_output_tokens=1024
)




In [ ]:
import json
from typing import List
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class CitizenQuestion(BaseModel):
    question: str = Field(description="Natural citizen question")
    scenario: str = Field(description="Citizen's situation")
    target_state: str = Field(description="Applicable state or All India")


class CitizenQuestionOutput(BaseModel):
    questions: List[CitizenQuestion]


question_generator = llm.with_structured_output(
    CitizenQuestionOutput
)


question_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are an expert in Indian government schemes and citizen assistance.

Your task is to generate realistic citizen questions using the
provided government scheme information.

STRICT RULES:

1. Never mention the scheme name or scheme ID.
2. Generate natural questions that citizens might ask.
3. Use only facts explicitly supported by the provided scheme data.
4. Never invent:
   - Age limits
   - Income limits
   - Caste or beneficiary restrictions
   - Loan amounts
   - Interest rates
   - Moratorium periods
   - Required documents
   - Eligibility conditions
5. Do not assume a citizen is eligible.
6. Do not introduce beneficiary categories not mentioned
   in the scheme information.
7. Questions must reflect the scheme's actual benefits.
8. Generate diverse questions, not simple paraphrases.
9. For State schemes, include the applicable state naturally
   in the question.
10. If a detail is missing, generate a question asking about
    that detail rather than inventing its answer.

IMPORTANT:
Every generated question must be traceable to the supplied
scheme information.
"""
    ),
    (
        "human",
        """
        Scheme information:
        {scheme_details}

        Geographic scope:
        {target_state}

        Generate exactly {num_questions} different questions.
        """
    )
])

question_generation_chain = (
    question_prompt | question_generator
)

print("Question generation chain initialized.")

Question generation chain initialized.


## Citizen Question Generation

Generate realistic citizen queries from government scheme information. Questions should describe the citizen's problem, needs, or circumstances without mentioning the scheme name. Preserve geographic and eligibility constraints so the generated questions can be used as retrieval benchmark queries.

In [19]:
test_task = generation_tasks[0]

scheme = test_task["scheme"]

scheme_details = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
    "exclusions": scheme.get("exclusions"),
    "brief_description": scheme.get("brief_description")
}

result = question_generation_chain.invoke({
    "scheme_details": json.dumps(
        scheme_details, ensure_ascii=False
    ),
    "target_state": test_task["target_state"],
    "num_questions": 5
})

for index, question in enumerate(result.questions, start=1):
    print(f"\nQuestion {index}: {question.question}")
    print("Scenario:", question.scenario)
    print("State:", question.target_state)


Question 1: I am a 30‑year‑old SC resident of Delhi earning around ₹1,00,000 a year. Can I get a loan to buy a small van for my transport business?
Scenario: SC individual, age 30, Delhi resident, family income ₹1,00,000/year, wants LMV for business.
State: Delhi

Question 2: My brother, who belongs to the minority community, is 42 and makes ₹5,50,000 annually. Is he eligible for a loan to purchase a motorbike for his delivery service?
Scenario: Minority individual, age 42, Delhi resident, family income ₹5,50,000/year, wants LMV.
State: Delhi

Question 3: I work as a Safai Karamchari in Delhi and have a valid commercial driving licence. Can I apply for a loan to buy a three‑wheel auto‑rickshaw?
Scenario: Safai Karamchari, Delhi resident, valid commercial licence, seeks LMV loan.
State: Delhi

Question 4: Is there any income ceiling for OBC applicants who want a loan for a light commercial vehicle, or can they apply regardless of earnings?
Scenario: OBC individual, Delhi resident, unsu

## Multi-Query Citizen Question Generation

Generate multiple diverse citizen queries from each selected government scheme. Keep the citizen's profile filters separate from the chat query. Generate profile attributes only when supported by the scheme's eligibility information. Preserve the source scheme ID and supporting fields for ground-truth evaluation.

In [22]:
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


# Structured output models
class CitizenProfile(BaseModel):
    state: Optional[str] = None
    caste_category: Optional[str] = None
    community_category: Optional[str] = None
    beneficiary_group: Optional[str] = None
    age: Optional[int] = None
    annual_family_income: Optional[float] = None
    scheme_category: Optional[str] = None


class CitizenQuestion(BaseModel):
    citizen_query: str
    profile: CitizenProfile
    intent: str
    supporting_fields: List[str]


class CitizenQuestionOutput(BaseModel):
    questions: List[CitizenQuestion]


# LLM structured output
question_generator = llm.with_structured_output(
    CitizenQuestionOutput
)


# Generation prompt
question_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an expert in Indian government schemes
        and citizen assistance platforms.

        Generate realistic citizen queries using the supplied
        government scheme information.

        PRODUCT CONTEXT:
        Citizens select their state, caste category, age
        and scheme category using profile filters BEFORE chatting.

        Therefore, citizen_query must describe only their
        actual problem or requirement.

        GENERATION RULES:

        1. Never mention the scheme name or scheme ID.
        2. Generate diverse real-world citizen problems.
        3. Do not repeat the citizen profile inside the query.
        4. Do not invent benefits, eligibility, income limits,
           age limits, documents or financial conditions.
        5. Profile caste_category and age must only be populated
           when supported by the scheme's eligibility information.
        6. Keep state applicability accurate.
        7. Use natural language that ordinary Indian citizens use.
        8. Avoid simple paraphrases of the same question.
        9. Generate different intents where supported:
           problem discovery, financial assistance, benefits,
           eligibility, application and required documents.
        10. Every query must be supported by the supplied
            scheme information.
        11. supporting_fields must identify the actual source
            fields used to construct the query.

        PROFILE AND CHAT SEPARATION:

1. State, caste, community, beneficiary group, age and income
   belong in the citizen profile.

2. Never repeat profile information in citizen_query.

3. citizen_query must contain only the citizen's actual
   problem, need or question.

4. Generate profile values only when supported by scheme
   eligibility information.

5. Do not invent exact income, loan amount, age or financial
   thresholds.

6. Use the scheme's original categories for scheme_category.
   Do not arbitrarily assign a different category.

7. Safai Karamchari and Minority must not automatically
   be classified as caste_category.

8. Every question must have supporting source fields.

9. Keep different question intents:
   - problem discovery
   - eligibility
   - benefits
   - documents
   - application process
   - financial terms
        """
    ),
    (
        "human",
        """
        SCHEME INFORMATION:

        {scheme_details}

        GEOGRAPHIC APPLICABILITY:
        {target_state}

        Generate exactly {num_questions} diverse citizen queries.

        Each query must represent a different genuine use case
        supported by this scheme.
        """
    )
])

question_generation_chain = (
    question_prompt | question_generator
)


# Test with one selected scheme
test_task = generation_tasks[0]
scheme = test_task["scheme"]

scheme_details = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "states": scheme.get("states"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
    "exclusions": scheme.get("exclusions"),
    "application_process": scheme.get("application_process"),
    "documents_required": scheme.get("documents_required"),
    "brief_description": scheme.get("brief_description")
}

result = question_generation_chain.invoke({
    "scheme_details": json.dumps(
        scheme_details, ensure_ascii=False
    ),
    "target_state": test_task["target_state"],
    "num_questions": 5
})


# Display generated queries
print("Source scheme:", scheme.get("name"))
print("Scheme ID:", scheme.get("scheme_id"))

for index, question in enumerate(result.questions, start=1):
    print(f"\n{'='*60}")
    print(f"QUESTION {index}")
    print("Citizen query:", question.citizen_query)
    print("Profile:", question.profile.model_dump())
    print("Intent:", question.intent)
    print("Supporting fields:", question.supporting_fields)

Source scheme: Transport Loan Scheme
Scheme ID: 3he_F5gB5uF80_MGg5Hz

QUESTION 1
Citizen query: I want to start a small transport business and need a loan to buy a light motor vehicle. What kind of loan options are available for this purpose?
Profile: {'state': 'Delhi', 'caste_category': None, 'community_category': None, 'beneficiary_group': 'individual', 'age': None, 'annual_family_income': None, 'scheme_category': 'Transport & Infrastructure'}
Intent: problem discovery
Supporting fields: ['benefits']

QUESTION 2
Citizen query: I belong to the OBC category, earn less than the income ceiling, and am 30 years old. Am I eligible for the loan?
Profile: {'state': 'Delhi', 'caste_category': 'Other Backward Classes', 'community_category': None, 'beneficiary_group': 'individual', 'age': 30, 'annual_family_income': None, 'scheme_category': 'Transport & Infrastructure'}
Intent: eligibility
Supporting fields: ['eligibility']

QUESTION 3
Citizen query: Could you tell me the list of documents I mu